In [8]:
import numpy as np
import pandas as pd

N_CUSTOMERS = 5000
rng = np.random.default_rng(123)

def lognorm_dollars(mean_log, sd_log):
    return float(np.round(np.exp(rng.normal(mean_log, sd_log)), 2))

def tier_from_wealth(w):
    if w < 0.55: return 0
    if w < 0.80: return 1
    if w < 0.95: return 2
    return 3

IBB_MULT   = {0: 0.70, 1: 1.00, 2: 1.80, 3: 3.60}
NIDDA_MULT = {0: 0.85, 1: 1.00, 2: 1.35, 3: 1.80}

LOAN_CAP = 200_000  # <-- your requirement

rows = []

for cid in range(1, N_CUSTOMERS + 1):
    wealth_proxy = rng.random()
    tier = tier_from_wealth(wealth_proxy)

    # ---------- DEPOSITS ----------
    deposit_types = {"Checking"}

    p_sav  = np.clip(0.70 + 0.10 * wealth_proxy, 0.55, 0.92)
    p_mm   = np.clip(0.08 + 0.28 * wealth_proxy, 0.05, 0.45)
    p_cd_s = np.clip(0.05 + 0.22 * wealth_proxy, 0.03, 0.30)
    p_cd_l = np.clip(0.02 + 0.12 * wealth_proxy, 0.01, 0.18)

    if rng.random() < p_sav:  deposit_types.add("Savings")
    if rng.random() < p_mm:   deposit_types.add("Money Market")
    if rng.random() < p_cd_s: deposit_types.add("CD Short Term")
    if rng.random() < p_cd_l: deposit_types.add("CD Long Term")

    for pt in deposit_types:
        if pt == "Checking":
            bal = lognorm_dollars(8.45, 0.80) * NIDDA_MULT[tier]
            dep_cat = "NIDDA"
        elif pt == "Savings":
            bal = lognorm_dollars(9.05, 0.95) * IBB_MULT[tier]
            dep_cat = "IBB"
        elif pt == "Money Market":
            bal = lognorm_dollars(9.55, 1.00) * IBB_MULT[tier]
            dep_cat = "IBB"
        elif pt == "CD Short Term":
            bal = lognorm_dollars(9.95, 1.05) * IBB_MULT[tier]
            dep_cat = "IBB"
        else:  # CD Long Term
            bal = lognorm_dollars(10.35, 1.15) * IBB_MULT[tier]
            dep_cat = "IBB"

        rows.append({
            "cust_id": cid,
            "product_group": "Deposit",
            "deposit_category": dep_cat,
            "product_type": pt,
            "balance": float(np.round(bal, 2)),
            "active_flag": "Y"
        })

    # ---------- CREDIT ----------
    p_cc = np.clip(0.42 + 0.22 * wealth_proxy, 0.18, 0.85)
    has_cc = rng.random() < p_cc
    cc_bal = lognorm_dollars(7.75, 0.95) if has_cc else 0.0

    rows.append({
        "cust_id": cid,
        "product_group": "Credit",
        "deposit_category": "N/A",
        "product_type": "Credit Card",
        "balance": float(np.round(cc_bal, 2)),
        "active_flag": "Y" if has_cc else "N"
    })

    # ---------- LOANS ----------
    p_home = np.clip(0.06 + 0.22 * wealth_proxy, 0.03, 0.28)
    p_auto = np.clip(0.09 + 0.08 * (1 - wealth_proxy), 0.05, 0.18)
    p_pl   = np.clip(0.06 + 0.07 * (1 - wealth_proxy), 0.03, 0.14)

    has_home = rng.random() < p_home
    has_auto = rng.random() < p_auto
    has_pl   = rng.random() < p_pl

    # Raw loan balances (pre-cap)
    home_bal = lognorm_dollars(12.2, 0.55) if has_home else 0.0   # lower than true mortgages, since you want <=200k
    auto_bal = lognorm_dollars(10.1, 0.55) if has_auto else 0.0
    pl_bal   = lognorm_dollars(9.4, 0.70)  if has_pl else 0.0

    # Apply cap per-loan
    home_bal = min(home_bal, LOAN_CAP)
    auto_bal = min(auto_bal, LOAN_CAP)
    pl_bal   = min(pl_bal, LOAN_CAP)

    rows.append({
        "cust_id": cid,
        "product_group": "Loan",
        "deposit_category": "N/A",
        "product_type": "Home Loan",
        "balance": float(np.round(home_bal, 2)),
        "active_flag": "Y" if has_home else "N"
    })
    rows.append({
        "cust_id": cid,
        "product_group": "Loan",
        "deposit_category": "N/A",
        "product_type": "Auto Loan",
        "balance": float(np.round(auto_bal, 2)),
        "active_flag": "Y" if has_auto else "N"
    })
    rows.append({
        "cust_id": cid,
        "product_group": "Loan",
        "deposit_category": "N/A",
        "product_type": "Personal Loan",
        "balance": float(np.round(pl_bal, 2)),
        "active_flag": "Y" if has_pl else "N"
    })

# Build table
fact_cust_product_v2 = pd.DataFrame(rows)

# --- Portfolio-level balancing: scale LOAN balances so total loans ≈ total deposits ---
active = fact_cust_product_v2.query("active_flag == 'Y'").copy()

dep_total = active.query("product_group == 'Deposit'")["balance"].sum()
loan_mask = (fact_cust_product_v2["active_flag"] == "Y") & (fact_cust_product_v2["product_group"] == "Loan")
loan_total = fact_cust_product_v2.loc[loan_mask, "balance"].sum()

# Avoid divide-by-zero if no loans
if loan_total > 0:
    scale = dep_total / loan_total
    # scale loan balances, then re-cap at 200k
    fact_cust_product_v2.loc[loan_mask, "balance"] = (
        fact_cust_product_v2.loc[loan_mask, "balance"] * scale
    ).clip(upper=LOAN_CAP).round(2)

# Active-only
fact_cust_product_active_v2 = fact_cust_product_v2.query("active_flag == 'Y'").copy()

# Sanity prints
dep_total2 = fact_cust_product_active_v2.query("product_group=='Deposit'")["balance"].sum()
loan_total2 = fact_cust_product_active_v2.query("product_group=='Loan'")["balance"].sum()

print("Deposits total:", round(dep_total2, 2))
print("Loans total   :", round(loan_total2, 2))
print("Loan/Deposit  :", round(loan_total2 / dep_total2, 3))
print("Max loan bal  :", fact_cust_product_active_v2.query("product_group=='Loan'")["balance"].max())

fact_cust_product_v2.head(10)

Deposits total: 185694319.61
Loans total   : 174757968.4
Loan/Deposit  : 0.941
Max loan bal  : 200000.0


,cust_id,product_group,deposit_category,product_type,balance,active_flag
0,1,Deposit,IBB,CD Short Term,38405.62,Y
1,1,Deposit,IBB,Money Market,7431.90,Y
2,1,Deposit,NIDDA,Checking,7212.43,Y
3,1,Deposit,IBB,Savings,6305.86,Y
4,1,Credit,N/A,Credit Card,0.00,N
5,1,Loan,N/A,Home Loan,0.00,N
6,1,Loan,N/A,Auto Loan,0.00,N
7,1,Loan,N/A,Personal Loan,0.00,N
8,2,Deposit,NIDDA,Checking,3096.55,Y
9,2,Credit,N/A,Credit Card,0.00,N


In [9]:
# Export active-only
fact_cust_product_active_v2.to_csv("/data/fact_cust_product.csv", index=False)